# Code Review Eval Guide

This guide shows how to evaluate AI code reviews with cached GitHub pull requests and the Evals API.


## Introduction

We will build three levels of increasing rigor:

- Level 1: a basic benchmark over raw cached PR records
- Level 2: the same benchmark over normalized PR records
- Level 3: pairwise comparison over normalized records

The key change in this version of the article is that the evaluation backend is the Evals API. The local helper code is responsible for fetching PRs, preparing JSONL datasets, and saving thin summaries after each run. The primary result surface is the Evals dashboard `report_url`.


## Setup

A reference implementation for this project lives in [examples/evals/codereview_evals](evals/codereview_evals).

Install the helper package:

```bash
cd examples/evals/codereview_evals
python -m venv .venv
source .venv/bin/activate
pip install -e .
```

Fetch pull requests once with GitHub CLI:

```bash
evalcr fetch-prs --repo openai/codex --limit 50
```

The rest of the workflow is dataset preparation plus Evals API runs:

```bash
evalcr prepare-dataset --level 1 --cache-key openai_codex
evalcr run-evals --level 1 --cache-key openai_codex
```

The CLI mirrors the Python SDK flow:

1. prepare JSONL rows from cached PRs
2. `client.files.create(...)`
3. `client.evals.create(...)` or reuse an existing eval
4. `client.evals.runs.create(...)`
5. inspect the async run and its `report_url`


## Level 1: Basic Benchmark

Level 1 keeps the data close to the raw cached pull request snapshot. The prepared record still normalizes the presentation a bit for stability, but it does not add a model-generated summary.

Each JSONL row stores stable PR metadata, merged state, a deterministic changed-file summary, a truncated diff excerpt, and normalized historical comments. The run prompt turns that record into a generated code review, and Evals graders score it for:

- correctness
- usefulness
- noise
- overall pass

A minimal SDK flow looks like this:

```python
from openai import OpenAI

client = OpenAI()

uploaded = client.files.create(
    file=open("data/prepared/openai_codex/level_1/benchmark.jsonl", "rb"),
    purpose="evals",
)

eval_obj = client.evals.create(
    name="code-review-evals-level-1",
    data_source_config={
        "type": "custom",
        "item_schema": {"type": "object", "properties": {...}, "required": [...]},
        "include_sample_schema": True,
    },
    testing_criteria=[...],
)

run = client.evals.runs.create(
    eval_obj.id,
    name="level-1-demo",
    data_source={
        "type": "responses",
        "model": "gpt-4.1",
        "input_messages": {
            "type": "template",
            "template": [
                {"role": "developer", "content": "...reviewer system prompt..."},
                {"role": "user", "content": "{{ item.review_input_text }}"},
            ],
        },
        "source": {"type": "file_id", "id": uploaded.id},
    },
)
```

After the run completes, the local helper computes only thin post-processing such as merged vs unmerged subgroup pass rates.


## Level 2: Benchmark With Normalized PR Records

Level 2 keeps the benchmark objective fixed and changes only the input representation. During preparation, we generate one cached `pr_brief` per pull request and reuse it across later runs.

That changes the review input from "raw-ish PR packet" to "normalized PR packet":

- raw PR metadata
- deterministic file summary
- truncated diff excerpt
- normalized historical comments
- cached model-generated `pr_brief`

This is useful because reviewer and grader behavior often becomes more stable once the task representation is cleaned up. The benchmark question stays the same, but the model sees a more consistent record.

Reference implementation: [examples/evals/codereview_evals/2_normalized_benchmark_harness](evals/codereview_evals/2_normalized_benchmark_harness)

```bash
evalcr prepare-dataset --level 2 --cache-key openai_codex
evalcr run-evals --level 2 --cache-key openai_codex
```

In article terms, this is where the story shifts from "how do I run a benchmark?" to "how do I make the benchmark itself more reliable?"


## Level 3: Pairwise Comparison On Normalized Records

Once the benchmark input is stable, we can compare review policies pairwise.

During preparation, Level 3 reuses the normalized PR record and caches two reviews for each PR:

- `baseline_review`
- `candidate_review`

The Evals run then judges those two reviews side by side and aggregates `baseline`, `candidate`, and `tie` outcomes after the run finishes.

Reference implementation: [examples/evals/codereview_evals/3_pairwise_harness](evals/codereview_evals/3_pairwise_harness)

```bash
evalcr prepare-dataset --level 3 --cache-key openai_codex
evalcr run-evals --level 3 --cache-key openai_codex
```

This framing is intentionally narrower than the old optimizer loop. Level 3 is now about expressing pairwise judging in Evals after normalizing the underlying PR representation, not about native prompt auto-optimization.
